<a href="https://colab.research.google.com/github/F1ameX/2025-ODS-NLP/blob/main/practice_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install and Import Libraries

In [1]:
!pip install --quiet catboost
!pip install --quiet gensim
!pip install --quiet nltk
!pip install --quiet pymystem3

In [2]:
import os
import re
import pandas as pd
import numpy as np

import nltk
from catboost import Pool, CatBoostClassifier
from pymystem3 import Mystem

## Read Data

In [84]:
path = '/content/drive/MyDrive/ods-nlp_src/practice_2/'
train_data = pd.read_csv(os.path.join(path, 'train.csv'))
test_data = pd.read_csv(os.path.join(path, 'test.csv'))
print(f'Number of rows and columns in the train data set: {train_data.shape}')
print(f'Number of rows and columns in the test data set: {test_data.shape}')
train_data.head()

Number of rows and columns in the train data set: (48665, 2)
Number of rows and columns in the test data set: (12167, 2)


,rate,text
0,4,Очень понравилось. Были в начале марта с соба...
1,5,В целом магазин устраивает.\nАссортимент позво...
2,5,"Очень хорошо что открылась 5 ка, теперь не над..."
3,3,Пятёрочка громко объявила о том как она заботи...
4,3,"Тесно, вечная сутолока, между рядами трудно ра..."


In [46]:
train_data.groupby('rate').describe()

text                               
      count unique                top freq
rate                                      
1      4138   4130             Грязно    3
2      2410   2407  Отстойный магазин    2
3      6126   6070          Нормально    7
4      9922   9763               Норм   14
5     26069  24804    Хороший магазин  107

## Preparing the data and creating Catboost model

In [47]:
train_data['text'].head(15)

,text
0,Очень понравилось. Были в начале марта с соба...
1,В целом магазин устраивает.\nАссортимент позво...
2,"Очень хорошо что открылась 5 ка, теперь не над..."
3,Пятёрочка громко объявила о том как она заботи...
4,"Тесно, вечная сутолока, между рядами трудно ра..."
5,Магазин в пешей доступности. После ремонта и р...
6,Магазин хороший цены и скидки нормальные токо ...
7,"Редко сюда забегаю. Маленький магазинчик, но э..."
8,Сложно найти в торговом центре. А магазин - норм)
9,После ремонта магазин в нутри стал ещё лучше. ...


In [85]:
def process_data(df):
    df['text'] = df['text'].str.lower()
    df['text'] = df['text'].apply(lambda x: re.sub(r'([,.!?;])', r' \1 ', x))
    df['text'] = df['text'].apply(lambda x: re.sub(r'[^\w\s]', '', x))
    df['text'] = df['text'].apply(lambda x: re.sub(r'[\d]', '', x))

    mystem = Mystem()
    df['text'] = df['text'].apply(lambda x: ' '.join(mystem.lemmatize(x)))
    df['text'] = df['text'].apply(lambda x: re.sub(r'[\n]', ' ', x))
    return df

In [86]:
train_data = process_data(train_data)
test_data = process_data(test_data)

In [87]:
train_data.head(15)

,rate,text
0,4,очень понравиться быть в начало ма...
1,5,в целое магазин устраивать ассортиме...
2,5,очень хорошо что открываться ка т...
3,3,пятерочка громко объявлять о то как ...
4,3,тесно вечный сутолока между ряд ...
5,4,магазин в пеший доступность после ...
6,5,магазин хороший цена и скидка нормал...
7,3,редко сюда забегать маленький магази...
8,5,сложно находить в торговый центр а...
9,4,после ремонт магазин в нутри станови...


In [88]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

corpus = train_data['text'].to_list() + test_data['text'].to_list()
tokenized_corpus = [simple_preprocess(sentence) for sentence in corpus]
w2v_model = Word2Vec(sentences = tokenized_corpus, vector_size = 100, min_count = 1)

In [89]:
train_data['tokens'] = train_data['text'].apply(lambda x: simple_preprocess(x))
test_data['tokens'] = test_data['text'].apply(lambda x: simple_preprocess(x))

In [112]:
train_data['word2vec'] = train_data['tokens'].apply(lambda x: [w2v_model.wv[word] for word in x if word in w2v_model.wv])
test_data['word2vec'] = train_data['tokens'].apply(lambda x: [w2v_model.wv[word] for word in x if word in w2v_model.wv])

In [113]:
train_data['word2vec'] = train_data['word2vec'].apply(lambda x: np.zeros(100) if len(x) == 0 else x)
test_data['word2vec'] = test_data['word2vec'].apply(lambda x: np.zeros(100) if len(x) == 0 else x)

In [114]:
X_train = np.vstack(train_data['word2vec'])
y_train = train_data['rate']

X_test = np.vstack(test_data['word2vec'])


model = CatBoostClassifier(
    iterations = 150,
    depth = 5,
    random_seed = 52,
)

model.fit(
    X_train,
    y_train,
    verbose=True
)

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 3300 and the array at index 1 has size 5000

## Predict

In [109]:
dataset_test = Pool(
    data = X_test,
)

predict_classes = model.predict(dataset_test)
predictions = predict_classes

## Create submission

In [110]:
sample_submission = pd.read_csv(os.path.join(path, 'sample_submission.csv'))
sample_submission['rate'] = predictions
sample_submission.head()

,index,rate
0,0,5
1,1,5
2,2,5
3,3,5
4,4,1


In [111]:
sample_submission.to_csv(os.path.join(path, 'submission.csv'), index=False)